# 12 — CUDA FMM memory usage

This notebook explains where the static FMM storage goes and compares the current CUDA-full and CUDA-partial layouts. It derives interaction counts from a real deterministic uniform tree and inspects canonical M2L matrices, but it never constructs a CUDA plan. Consequently, even configurations predicted to exceed device memory can be studied safely. If the active Python extension does not yet contain `static_m2l_matrix`, run `cmake --fresh --preset notebooks` followed by `cmake --build --preset notebooks -j` from the repository root, then restart the kernel.

Here, *space* means persistent host or device memory. It does not mean the geometric volume occupied by the particles.

In [ ]:
# All user-editable parameters are collected here.
N_PARTICLES = 10_000
EXPANSION_ORDER = 6
TREE_DEPTHS = [2, 3, 4]
PARTICLE_COUNTS = [1_000, 3_000, 10_000, 30_000, 100_000]
REFERENCE_GPU_GIB = 8.0
RANDOM_SEED = 314159


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

try:
    from fmm_memory import (
        estimate_source_point_storage, coefficient_count,
        P2P_BLOCK_BYTES, STATIC_ENTRY_BYTES,
    )
except ModuleNotFoundError:
    from examples.notebooks.fmm_memory import (
        estimate_source_point_storage, coefficient_count,
        P2P_BLOCK_BYTES, STATIC_ENTRY_BYTES,
    )

rng = np.random.default_rng(RANDOM_SEED)
maximum_particles = max(max(PARTICLE_COUNTS), N_PARTICLES)
master_positions = rng.uniform(-0.95, 0.95, size=(maximum_particles, 3))
positions = master_positions[:N_PARTICLES]
GIB = 1024**3
GB = 1000**3
print(f"Order {EXPANSION_ORDER} has {coefficient_count(EXPANSION_ORDER)} coefficients.")
print(f"Static entry: {STATIC_ENTRY_BYTES} bytes; P2P tensor block: {P2P_BLOCK_BYTES} bytes.")

## What is stored

For Cartesian order $p$, the coefficient count is $n_p=\binom{p+3}{3}$. At order 6, $n_p=84$ and a dense M2L matrix contains 7,056 doubles.

Both CUDA paths store list-1 P2P as one 56-byte symmetric dipole tensor block per ordered target–source pair, plus row offsets and dynamic moment/field buffers. At shallow depth this term behaves approximately as $27N^2/8^d$.

**CUDA-full** uploads P2M, M2M, M2L, L2L, L2P, and P2P. Every retained M2L matrix value is expanded into a 16-byte `(output, input, value)` entry for every box interaction. Evaluation then needs only moments uploaded and final fields downloaded, but static M2L storage grows approximately as the number of occupied boxes.

**CUDA-partial** keeps P2M, M2M, L2L, and L2P on the CPU. It uploads one dense M2L matrix per transfer class, compact source/target node indices, packed coefficients, and gathered/translated work arrays. Its matrix values are not replicated per interaction, but multipoles and locals cross the device boundary each evaluation and the CPU stages remain part of the critical path.

In [ ]:
selected = [
    estimate_source_point_storage(positions, EXPANSION_ORDER, depth)
    for depth in TREE_DEPTHS
]
print(
    f"{'Depth':>5s} {'Nodes':>8s} {'Classes':>9s} {'M2L pairs':>12s} "
    f"{'P2P pairs':>12s} {'Host GiB':>10s} {'Partial GiB':>12s} {'Full GiB':>10s}"
)
for estimate in selected:
    print(
        f"{estimate.depth:5d} {estimate.nodes:8d} "
        f"{estimate.transfer_classes:9d} {estimate.m2l_interactions:12d} "
        f"{estimate.p2p_pairs:12d} {estimate.host_static_bytes/GIB:10.3f} "
        f"{estimate.cuda_partial_bytes/GIB:12.3f} {estimate.cuda_full_bytes/GIB:10.3f}"
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for axis, attribute, title in [
    (axes[0], "cuda_partial", "CUDA-partial device storage"),
    (axes[1], "cuda_full", "CUDA-full device storage"),
]:
    bottoms = np.zeros(len(selected))
    component_names = list(getattr(selected[0], attribute))
    for component in component_names:
        values = np.array([getattr(item, attribute)[component] / GIB for item in selected])
        axis.bar(TREE_DEPTHS, values, bottom=bottoms, label=component)
        bottoms += values
    axis.axhline(REFERENCE_GPU_GIB, color="black", linestyle="--", label="Reference budget")
    axis.set_title(title)
    axis.set_xlabel("Tree depth")
    axis.set_xticks(TREE_DEPTHS)
    axis.set_yscale("log")
    axis.grid(True, axis="y", which="both", alpha=0.25)
    axis.legend(fontsize=7)
axes[0].set_ylabel("Estimated persistent device storage (GiB)")
fig.suptitle(f"Storage breakdown: N={N_PARTICLES:,}, order={EXPANSION_ORDER}")
fig.tight_layout()
plt.show()

## Why depth moves the bottleneck

Increasing depth divides each leaf into eight. This normally reduces list-1 P2P pairs by roughly eight, but it creates up to eight times as many occupied target boxes and therefore many more M2L interactions. CUDA-full replicates M2L values across those interactions; CUDA-partial reuses transfer-class matrices and grows mainly through coefficient work buffers. The memory-optimal depth is therefore a balance, not simply the deepest possible tree.

In [ ]:
scaling = {}
for depth in TREE_DEPTHS:
    scaling[depth] = [
        estimate_source_point_storage(
            master_positions[:count], EXPANSION_ORDER, depth
        )
        for count in PARTICLE_COUNTS
    ]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for axis, total_name, title in [
    (axes[0], "cuda_partial_bytes", "CUDA-partial"),
    (axes[1], "cuda_full_bytes", "CUDA-full"),
]:
    for depth, estimates in scaling.items():
        axis.plot(
            PARTICLE_COUNTS,
            [getattr(item, total_name) / GIB for item in estimates],
            marker="o", label=f"depth {depth}",
        )
    axis.axhline(REFERENCE_GPU_GIB, color="black", linestyle="--", label="Reference budget")
    axis.set_xscale("log")
    axis.set_yscale("log")
    axis.set_xlabel("Particles")
    axis.set_title(title)
    axis.grid(True, which="both", alpha=0.25)
    axis.legend()
axes[0].set_ylabel("Estimated persistent device storage (GiB)")
fig.suptitle(f"Particle and depth scaling at order {EXPANSION_ORDER}")
fig.tight_layout()
plt.show()

print(f"Configurations exceeding the {REFERENCE_GPU_GIB:g} GiB reference budget:")
for depth, estimates in scaling.items():
    for item in estimates:
        flags = []
        if item.cuda_partial_bytes > REFERENCE_GPU_GIB * GIB:
            flags.append("partial")
        if item.cuda_full_bytes > REFERENCE_GPU_GIB * GIB:
            flags.append("full")
        if flags:
            print(f"  N={item.particles:>7,d}, depth={depth}: {', '.join(flags)}")

In [ ]:
print("Host-side static plan retained by the cdfmm object:")
for item in selected:
    print(f"  depth {item.depth}: {item.host_static_bytes/GIB:.3f} GiB")

host_components = list(selected[0].host_static)
bottoms = np.zeros(len(selected))
plt.figure(figsize=(9, 5))
for component in host_components:
    values = np.array([item.host_static[component] / GIB for item in selected])
    plt.bar(TREE_DEPTHS, values, bottom=bottoms, label=component)
    bottoms += values
plt.yscale("log")
plt.xticks(TREE_DEPTHS)
plt.xlabel("Tree depth")
plt.ylabel("Estimated host plan storage (GiB)")
plt.title("CPU static-plan storage retained alongside CUDA plans")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Practical interpretation

| Backend | Advantages | Costs |
|---|---|---|
| CUDA-full | All six FMM stages remain on the GPU; normal evaluations transfer only moments and final fields; lowest steady-state latency when the plan fits. | Replicates nonzero M2L values for every interaction; large construction uploads; depth can increase storage dramatically. |
| CUDA-partial | Stores each dense M2L matrix once per transfer class; substantially smaller static M2L representation; CPU operators remain directly inspectable. | Retains CPU stages; requires packed multipole H2D and local D2H transfers; gather/scatter work buffers and synchronisation remain. |

The estimates follow the current C++ structures and the exact interaction lists for these coordinates. They exclude allocator rounding and metadata, CUDA context/library allocations, pinned host staging buffers, Python objects, plotting arrays, and memory used by other processes. A plan whose estimate is close to nominal GPU capacity can therefore still fail. Host and device totals describe different physical memory domains and should not be added when checking a device limit.